# 2-layer model tests

Tests analogous to the 1-layer notebook, exercising the multi-layer extensions:
1. Smoke test (all setup methods + run with `nk=2`).
2. Symmetric layers stay symmetric — equal `Ho`, equal IC, no asymmetric forcing.
3. Inertial oscillations.
4. Burger term advection.
5. Lateral viscosity decay.
6. Barotropic gravity wave equivalent to 1-layer (`g=[g, 0]`).
7. Internal (baroclinic) gravity wave (`g=[g, g']`).

**Note on Step 4:** wind stress and bottom drag are currently broadcast onto every layer. After the planned vertical-stress-divergence refactor, wind will only force the top layer and linear drag will only act on the bottom layer. A few of the 1-layer↔2-layer numerical comparisons below will only become bit-meaningful after that change.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from OSSWEM import SSWEM

## 1. Smoke test
Exercise every setup method with `nk=2`.

In [ ]:
M = SSWEM(16,
          [10., 0.02], # g, g' [m s-2]
          [200., 800.], # Nominal layer thickness [m]
          1000.e3, # Domain width [m]
          1.e-4, # Coriolis [s-1]
          2.e-11, # df/dy [m-1 s-1]
          1.e-3, # Bottom drag rate [m s-1]
          1.e-1, # Lateral viscosity [m2 s-1]
          h_relax=1e-4) # Forcing rate for eta [s-1]
M.resting_state()
M.flat_topog()
M.bowl_topog()
M.zero_forcing()
M.gyre_forcing()
M.channel_forcing()
M.set_h_forcing(0.1)
M.perturb_h(0.01, 50.e3, 500.e3, k=0)
M.perturb_h(0.01, 50.e3, 500.e3, k=1)
M.step(50.)
u, v, h, t = M.run(10., 1, 2)
print('shapes:  u', u.shape, '  h', h.shape, '  eta()', M.eta().shape)
print('eta(k=0) shape (free surface):', M.eta(k=0).shape)
print('q() shape:', M.q().shape)

## 2. Symmetric layers stay symmetric
Equal `Ho` per layer, `g'=0` (so the only pressure-gradient force is from the free surface, identical on both layers), identical perturbation in each layer. The two layers should remain identical for all time — exercises Coriolis, advection, viscosity, and continuity in tandem.

In [ ]:
M = SSWEM(32,
          [10., 0.], # g, g' [m s-2]
          [500., 500.], # Nominal layer thickness [m]
          1000.e3, # Domain width [m]
          1.e-4, # Coriolis [s-1]
          2.e-11, # df/dy [m-1 s-1]
          0., # Bottom drag rate [m s-1]
          1.e-1) # Lateral viscosity [m2 s-1]
# Identical perturbation in each layer
M.perturb_h(0.01, 50.e3, 500.e3, y0=500.e3, k=0)
M.perturb_h(0.01, 50.e3, 500.e3, y0=500.e3, k=1)
u, v, h, t = M.run(50., 5, 20)
for n, name in [(h, 'h'), (u, 'u'), (v, 'v')]:
    diff = np.max(np.abs(n[:, 0] - n[:, 1]))
    scale = max(np.abs(n).max(), 1e-300)
    print(f'  max |{name}[0] - {name}[1]| = {diff:.2e}   (scale {scale:.2e})')

## 3. Inertial oscillations
Pure inertial response (no gravity, no drag) on a single column with `nk=2`. Both layers should stay identical and the implicit Coriolis solve should be stable + convergent across `f·Δt` ratios.

In [ ]:
def run_inertial(dt, nsteps):
    M = SSWEM(1,
              [0., 0.], # g, g' [m s-2]
              [500., 500.], # Nominal layer thickness [m]
              1000.e3, # Domain width [m]
              1.e-4, # Coriolis [s-1]
              0., # df/dy [m-1 s-1]
              0., # Bottom drag rate [m s-1]
              0) # Lateral viscosity [m2 s-1]
    M.gyre_forcing()
    return M, *M.run(dt, 1, nsteps)

fig, ax = plt.subplots()
for dt, nsteps, color, label in [(500., 1000, 'C0', r'$f\Delta t=1/20$'),
                                  (5000., 100, 'C1', r'$f\Delta t=1/2$'),
                                  (20000., 25, 'C2', r'$f\Delta t=2$')]:
    M, u, v, h, time = run_inertial(dt, nsteps)
    ax.plot(time/(2*np.pi*M.fo), u[:,0,0,0], color, label=f'{label} (top)')
    ax.plot(time/(2*np.pi*M.fo), u[:,1,0,0], '--'+color, label=f'{label} (bot)')
    print(f'{label}: layers identical?', np.allclose(u[:,0], u[:,1]))
ax.set_xlabel(r'$t / (2\pi f)$'); ax.set_ylabel('u'); ax.legend(fontsize=8);

## 4. Burger term
1-D advection of $u$ with no gravity or rotation. Initialize identical $u$ profiles in both layers — they should stay identical.

In [ ]:
M = SSWEM(100,
          [0., 0.], # g, g' [m s-2]
          [500., 500.], # Nominal layer thickness [m]
          1000., # Domain width [m]
          0., # Coriolis [s-1]
          0., # df/dy [m-1 s-1]
          0., # Bottom drag rate [m s-1]
          0) # Lateral viscosity [m2 s-1]
u_init = np.sin(2*np.pi*M.xu/M.Lx) + 0.2
M.u[0] = u_init
M.u[1] = u_init
u, v, h, time = M.run(5., 10, 10)
fig, ax = plt.subplots()
ax.plot(M.xq1, u[:,0,0,:].T)
ax.plot(M.xq1, u[:,1,0,:].T, '--', alpha=0.5)
ax.set_xlabel(r'$x$ [m]'); ax.set_ylabel(r'$u$ [m s$^{-1}$]')
ax.set_title('top (solid) and bottom (dashed) layers')
print('layers identical:', np.allclose(u[:,0], u[:,1]))

## 5. Lateral viscosity decay
Highest-frequency mode (alternating $\pm 1$) decays under lateral viscosity. Both layers identical.

In [ ]:
M = SSWEM(10,
          [0., 0.], # g, g' [m s-2]
          [500., 500.], # Nominal layer thickness [m]
          1000., # Domain width [m]
          0., # Coriolis [s-1]
          0., # df/dy [m-1 s-1]
          0., # Bottom drag rate [m s-1]
          2e2) # Lateral viscosity [m2 s-1]
u_init = np.cos(np.pi*M.xu/M.dx)
M.u[0] = u_init
M.u[1] = u_init
u, v, h, time = M.run(5., 1, 2)
fig, ax = plt.subplots()
ax.plot(M.xq1, u[:,0,0,:].T)
ax.plot(M.xq1, u[:,1,0,:].T, '--', alpha=0.5)
ax.set_xlabel(r'$x$ [m]'); ax.set_ylabel(r'$u$ [m s$^{-1}$]')
print('layers identical:', np.allclose(u[:,0], u[:,1]))

## 6. Barotropic gravity wave (vs. 1-layer)
With `g=[g_full, 0]` (zero reduced gravity at the internal interface) the only pressure gradient comes from the free surface, identical on both layers. With `Ho=[H/2, H/2]`, total depth `H`, the external mode wave speed `cg = √(g·H)` matches the equivalent 1-layer model. We perturb each layer by `δ/2` (so the total column perturbation matches a 1-layer `δ` perturbation) and compare Hovmöller plots.

In [ ]:
M1 = SSWEM(80,
           10., # Gravity [m s-2]
           250., # Nominal layer thickness [m]
           1000.e3, # Domain width [m]
           0., # Coriolis [s-1]
           0., # df/dy [m-1 s-1]
           0., # Bottom drag rate [m s-1]
           0) # Lateral viscosity [m2 s-1]
M1.perturb_h(0.01, 50.e3, M1.Lx/2, k=0)
u1, v1, h1, time1 = M1.run(200., 1, 50)

M2 = SSWEM(80,
           [10., 0.], # g, g' [m s-2]
           [125., 125.], # Nominal layer thickness [m]
           1000.e3, # Domain width [m]
           0., # Coriolis [s-1]
           0., # df/dy [m-1 s-1]
           0., # Bottom drag rate [m s-1]
           0) # Lateral viscosity [m2 s-1]
M2.perturb_h(0.005, 50.e3, M1.Lx/2, k=0)
M2.perturb_h(0.005, 50.e3, M1.Lx/2, k=1)
u2, v2, h2, time2 = M2.run(200., 1, 50)

# Free-surface eta over time (per snapshot, top interface)
eta1 = np.array([M1.eta(h1[t], k=0) for t in range(len(time1))])
eta2 = np.array([M2.eta(h2[t], k=0) for t in range(len(time2))])

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True, sharex=True)
ax[0].contourf(M1.xh1/1e3, time1/3600, eta1[:, 0, :])
ax[0].plot((M1.Lx/2 + M1.cg*time1)/1e3, time1/3600, '--w')
ax[0].plot((M1.Lx/2 - M1.cg*time1)/1e3, time1/3600, '--w')
ax[0].set_title(f'1-layer  cg={M1.cg:.0f} m/s')
ax[0].set_xlabel('x [km]'); ax[0].set_ylabel('t [h]')
ax[1].contourf(M2.xh1/1e3, time2/3600, eta2[:, 0, :])
ax[1].plot((M2.Lx/2 + M2.cg*time2)/1e3, time2/3600, '--w')
ax[1].plot((M2.Lx/2 - M2.cg*time2)/1e3, time2/3600, '--w')
ax[1].set_title(f'2-layer  cg={M2.cg:.0f} m/s (g=[10, 0])')
ax[1].set_xlabel('x [km]')
print('cg match:', M1.cg == M2.cg)
print('peak |eta1 - eta2|:', np.max(np.abs(eta1 - eta2)))
print('peak  eta1 magnitude:', np.abs(eta1).max())

## 7. Internal gravity wave
2-layer specific test (no 1-layer analog). Perturb the internal interface only — depress it under the bump by raising `h_0` and lowering `h_1` by the same amount, leaving the free surface flat. The disturbance should propagate as an internal wave at
$$c_i = \sqrt{g'\,\frac{H_1 H_2}{H_1+H_2}}.$$

In [ ]:
M = SSWEM(80,
          [10., 2.], # g, g' [m s-2]
          [500., 500.], # Nominal layer thickness [m]
          1000.e3, # Domain width [m]
          0., # Coriolis [s-1]
          0., # df/dy [m-1 s-1]
          0., # Bottom drag rate [m s-1]
          0) # Lateral viscosity [m2 s-1]
delta = 5.
M.perturb_h(+delta, 50.e3, M.Lx/2, k=0)
M.perturb_h(-delta, 50.e3, M.Lx/2, k=1)
print('IC: free surface max =', np.abs(M.eta(k=0)).max(),
      '  interface max =', np.abs(M.eta(k=1) - (-M.D + M.Ho[1])).max())

gp = M.g[1]; H1, H2 = M.Ho
c_i = np.sqrt(gp * H1*H2 / (H1+H2))
print(f'c_i = {c_i:.3f} m/s   period for L={M.Lx/1e3:.0f}km: {M.Lx/c_i/3600:.1f} h')

u, v, h, time = M.run(50., 2, 200)
interface = np.array([M.eta(h[t], k=1) for t in range(len(time))])
free_surf = np.array([M.eta(h[t], k=0) for t in range(len(time))])

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True, sharex=True)
ax[0].contourf(M.xh1/1e3, time/3600, free_surf[:, 0, :], cmap='RdBu_r')
ax[0].plot((M.Lx/2 + M.cg*time)/1e3, time/3600, '--k', label='external cg')
ax[0].plot((M.Lx/2 - M.cg*time)/1e3, time/3600, '--k')
ax[0].plot((M.Lx/2 + c_i*time)/1e3, time/3600, ':k', label=f'$c_i={c_i:.2f}$ m/s')
ax[0].plot((M.Lx/2 - c_i*time)/1e3, time/3600, ':k')
ax[0].set_xlim(0, M.Lx/1e3)
ax[0].set_title('Free surface  $\\eta_{1/2}$'); ax[0].set_xlabel('x [km]'); ax[0].set_ylabel('t [h]')
ax[0].legend(loc='upper right', fontsize=8)
ax[1].contourf(M.xh1/1e3, time/3600, interface[:, 0, :] - (-M.D[0,0] + M.Ho[1]), cmap='RdBu_r')
ax[1].plot((M.Lx/2 + c_i*time)/1e3, time/3600, ':k', label=f'$c_i={c_i:.2f}$ m/s')
ax[1].plot((M.Lx/2 - c_i*time)/1e3, time/3600, ':k')
ax[1].set_xlim(0, M.Lx/1e3)
ax[1].set_title('Internal interface  $\\eta_{3/2}$'); ax[1].set_xlabel('x [km]')
ax[1].legend(loc='upper right', fontsize=8);